In [37]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import confusion_matrix, accuracy_score, ConfusionMatrixDisplay, precision_score, recall_score
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.model_selection import cross_val_score, KFold
from sklearn.naive_bayes import BernoulliNB, MultinomialNB, GaussianNB
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

In [38]:
df = pd.read_csv('processed-tweets.csv')

In [39]:
df.shape

(1583691, 3)

In [40]:
df.sample(5)

,target,text,processed_text
623521,0,I Miss You Tommy!,miss tommi
1401610,1,@kenmitch least you have a done album.,least done album
449018,0,"@gaylejack I'm around, but not blipping much l...",around blip much late sorri
1076734,1,Sleeping finally. &lt;3,sleep final lt 3
120422,0,@legend1dflight Nah man...you don't fuck w/me ...,nah man fuck w


In [41]:
df.isnull().sum()

target               0
text                 0
processed_text    7487
dtype: int64

In [42]:
df.dropna(inplace = True)

In [43]:
df.isnull().sum()

target            0
text              0
processed_text    0
dtype: int64

In [44]:
X = df['processed_text']
y = df['target']

In [45]:
X

0                    bummer shoulda got david carr third day
1          upset updat facebook text might cri result sch...
2            dive mani time ball manag save 50 rest go bound
3                            whole bodi feel itchi like fire
4                                              behav mad see
                                 ...                        
1583686                           woke school best feel ever
1583687            thewdb com cool hear old walt interview â
1583688                         readi mojo makeov ask detail
1583689    happi 38th birthday boo alll time tupac amaru ...
1583690                                 happi charitytuesday
Name: processed_text, Length: 1576204, dtype: object

In [46]:
y

0          0
1          0
2          0
3          0
4          0
          ..
1583686    1
1583687    1
1583688    1
1583689    1
1583690    1
Name: target, Length: 1576204, dtype: int64

## vectorize text

In [47]:
tfidf = TfidfVectorizer(stop_words = 'english',
                        dtype = np.float32)

In [48]:
X_tfidf = tfidf.fit_transform(X.values.astype(str))

## Model Building

In [13]:
mnb = MultinomialNB()

In [14]:
kfold = KFold(n_splits = 10, shuffle = True, random_state = 42)
scores = cross_val_score(mnb, X_tfidf, y, cv = kfold, scoring = 'accuracy')

In [15]:
scores.mean(), scores.std()

(np.float64(0.7535528391200554), np.float64(0.0011202439566999499))

In [16]:
def scorer(model_name, model):

    output = []

    output.append(model_name)
    
    kfold = KFold(n_splits = 10, shuffle = True, random_state = 42)
    scores = cross_val_score(model, X_tfidf, y, cv = kfold, scoring = 'accuracy')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size = 0.2, random_state = 42)

    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    output.append(accuracy_score(y_test, y_pred))
    output.append(confusion_matrix(y_test, y_pred))

    return output

In [17]:
model_dict = {
    'logisticRegression':LogisticRegression(max_iter = 1000, n_jobs = -1),
    'MultinomialNB':MultinomialNB(),
    'BernoulliNB':BernoulliNB(),
    'LinearSVC':LinearSVC(),
    'SGDClassifier':SGDClassifier(loss = 'log_loss', max_iter = 1000, n_jobs = -1),
    'XGBClassifier':XGBClassifier(n_estimators = 300, n_jobs = -1)
}

In [18]:
model_output = []

for model_name, model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [19]:
model_df = pd.DataFrame(model_output, columns = ['name', 'cv_accuracy', 'test_accuracy', 'confusion_matrix'])

In [20]:
model_df

,name,cv_accuracy,test_accuracy,confusion_matrix
0,logisticRegression,0.769541,0.768469,"[[116310, 40430], [32558, 125943]]"
1,MultinomialNB,0.753553,0.753024,"[[117834, 38906], [38951, 119550]]"
2,BernoulliNB,0.761735,0.760777,"[[117810, 38930], [36483, 122018]]"
3,LinearSVC,0.763222,0.761741,"[[115367, 41373], [33736, 124765]]"
4,SGDClassifier,0.752182,0.751482,"[[112184, 44556], [33787, 124714]]"
5,XGBClassifier,0.754500,0.753700,"[[107417, 49323], [28321, 130180]]"


In [21]:
model_df = model_df.sort_values(by = 'cv_accuracy', ascending = False)

In [22]:
model_df

,name,cv_accuracy,test_accuracy,confusion_matrix
0,logisticRegression,0.769541,0.768469,"[[116310, 40430], [32558, 125943]]"
3,LinearSVC,0.763222,0.761741,"[[115367, 41373], [33736, 124765]]"
2,BernoulliNB,0.761735,0.760777,"[[117810, 38930], [36483, 122018]]"
5,XGBClassifier,0.754500,0.753700,"[[107417, 49323], [28321, 130180]]"
1,MultinomialNB,0.753553,0.753024,"[[117834, 38906], [38951, 119550]]"
4,SGDClassifier,0.752182,0.751482,"[[112184, 44556], [33787, 124714]]"


In [23]:
for row in model_df.itertuples(index = False, name = 'ModelMetrics'):
    model_name = row.name
    matrix = row.confusion_matrix
    print(f"Model: {model_name}", "\n",matrix)
    print('-----')

Model: logisticRegression 
 [[116310  40430]
 [ 32558 125943]]
-----
Model: LinearSVC 
 [[115367  41373]
 [ 33736 124765]]
-----
Model: BernoulliNB 
 [[117810  38930]
 [ 36483 122018]]
-----
Model: XGBClassifier 
 [[107417  49323]
 [ 28321 130180]]
-----
Model: MultinomialNB 
 [[117834  38906]
 [ 38951 119550]]
-----
Model: SGDClassifier 
 [[112184  44556]
 [ 33787 124714]]
-----


## Training Model On LogisticRegression    

In [11]:
tfidf = TfidfVectorizer(stop_words = 'english',
                        dtype = np.float32)
X_tfidf = tfidf.fit_transform(X.values.astype(str))

In [49]:
model = LogisticRegression(max_iter = 1000, class_weight = 'balanced', solver = 'saga', n_jobs = -1)

In [50]:
kfold = KFold(n_splits = 10, shuffle = True, random_state = 42)
scores = cross_val_score(model, X_tfidf, y, cv = kfold, scoring = 'accuracy', n_jobs = -1)

In [51]:
scores.mean(), scores.std()

(np.float64(0.7703425452804741), np.float64(0.0010446648607333566))

In [52]:
model.fit(X_tfidf, y)

LogisticRegression(class_weight='balanced', max_iter=1000, n_jobs=-1,
                   solver='saga')

In [24]:
import pickle
filename = 'tweets-model.pkl'
with open(filename, 'wb') as file:
    pickle.dump(model, file)

In [27]:
with open("tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

## Testing Model

In [53]:
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size = 0.2, random_state = 42)

In [54]:
model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, n_jobs=-1,
                   solver='saga')

In [56]:
y_pred = model.predict(X_test)

In [57]:
print('Confusion Matrix :\n', confusion_matrix(y_test, y_pred))
print('Accuracy Score :', accuracy_score(y_test, y_pred)*100)
print('Precision Score :', precision_score(y_test, y_pred)*100)
print('Recall Score :', recall_score(y_test, y_pred)*100)

Confusion Matrix :
 [[116552  40188]
 [ 32594 125907]]
Accuracy Score : 76.91226712261413
Precision Score : 75.80420843493182
Recall Score : 79.4360918858556
